# 01 — Tổng quan và kiểm tra dữ liệu Valorant

Mục tiêu của notebook:

1. Kiểm tra quy mô, phạm vi thời gian và mức độ thiếu dữ liệu.
2. Hiểu vì sao KAST bị loại khỏi mô hình.
3. Chạy quy trình làm sạch và xem số ván/người chơi còn lại.

Dữ liệu gốc: `data/raw/valorant.sqlite`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from valorant_mining.config import DB_PATH, OUTPUT_TABLES
from valorant_mining.db import read_sql, table_counts

print("DB:", DB_PATH, DB_PATH.exists())
table_counts()


DB: /Users/minhduc/work/valorant-mining/data/raw/valorant.sqlite True


,table,rows
0,Matches,7818
1,Games,15888
2,Game_Scoreboard,157939
3,Game_Rounds,15531


## 1. Quy mô dữ liệu và phân bố bản đồ

In [2]:
maps = read_sql("SELECT Map, COUNT(*) AS games FROM Games GROUP BY Map ORDER BY games DESC")
dates = read_sql("SELECT MIN(Date) AS first_date, MAX(Date) AS last_date FROM Matches")
display(dates)
display(maps)


,first_date,last_date
0,2020-05-02 13:00:00,2022-01-08 15:30:00


,Map,games
0,Ascent,3866
1,Haven,3487
2,Bind,3355
3,Split,2408
4,Icebox,1944
5,Breeze,661
6,Fracture,107
7,TBD,60


## 2. Thiếu dữ liệu

KAST gần như thiếu toàn bộ, nên không thể dùng làm biến trung tâm. Không thay giá trị thiếu bằng 0 vì sẽ tạo ra tín hiệu giả.


In [3]:
missing = read_sql('''
SELECT
  SUM(PlayerID IS NULL) AS player_id_null,
  SUM(Agent IS NULL OR TRIM(Agent) = '') AS agent_blank,
  SUM(ACS IS NULL) AS acs_null,
  SUM(KAST_Percent IS NULL) AS kast_null,
  SUM(Num_2Ks IS NULL) AS multikill_null,
  COUNT(*) AS rows
FROM Game_Scoreboard
''')
missing


,player_id_null,agent_blank,acs_null,kast_null,multikill_null,rows
0,87,4100,530,154572,10195,157939


## 3. Làm sạch và kiểm tra đội hình

Quy tắc: mỗi đội đúng 5 người chơi, 5 Agent khác nhau; mỗi ván đúng 2 đội; số round lấy từ RoundHistory nếu hợp lệ.


In [4]:
from valorant_mining.cleaning import build_clean_data

clean = build_clean_data()
pd.Series(clean.audit).to_frame("value")


,value
scoreboard_rows_raw,157939
dropped_missing_player_id,87
dropped_bad_agent_or_team,6265
dropped_duplicate_player_game,4
scoreboard_rows_clean,151583
games_raw,15888
games_with_round_history,14854
round_history_inconsistent,7
games_with_valid_rosters,14016
games_after_round_and_map_filter,14005


In [5]:
comps = clean.compositions
summary = (comps.groupby("Map")
           .agg(teams=("GameID", "size"), games=("GameID", "nunique"), labeled=("won", "count"))
           .sort_values("teams", ascending=False))
summary


,teams,games,labeled
Map,,,
Ascent,6806,3403,6652
Haven,6216,3108,6036
Bind,5896,2948,5730
Split,4224,2112,4096
Icebox,3494,1747,3494
Breeze,1176,588,1176
Fracture,198,99,198


## 4. Nhận xét

- Chỉ khoảng 88% ván tạo được đội hình 2 × 5 người và 5 Agent.
- KAST thiếu gần như toàn bộ → loại.
- Breeze/Fracture có ít mẫu; luật của hai map này cần thận trọng.
- Bước tiếp theo: `02_phan_cum_nguoi_choi.ipynb`.
